# Build the E2EE Chat APK in Colab
Run the cells **top to bottom**. Total time is roughly 20-40 minutes on the first run.

You will upload 5 files from your `Chat_Application` folder: `main.py`, `crypto.py` (the **new** pycryptodome version), `storage.py`, `protocol.py`, `mobile_client.py`.

## 1. Install build tools (about 3-5 min)

In [ ]:
%%bash
sudo apt-get update -qq
sudo apt-get install -y -qq git zip unzip openjdk-17-jdk autoconf libtool pkg-config \
    zlib1g-dev libncurses-dev cmake libffi-dev libssl-dev build-essential > /dev/null
pip install -q buildozer cython==0.29.36 setuptools
java -version 2>&1 | head -1

## 2. Upload your 5 files
In the file picker, select all five at once.

In [ ]:
import os, shutil
from google.colab import files

shutil.rmtree('/content/app', ignore_errors=True)
os.makedirs('/content/app')
os.chdir('/content/app')
files.upload()   # choose: main.py, crypto.py, storage.py, protocol.py, mobile_client.py

required = ['main.py', 'crypto.py', 'storage.py', 'protocol.py', 'mobile_client.py']
missing = [f for f in required if not os.path.exists(f)]
assert not missing, f"Missing files: {missing}"
assert 'cryptography' not in open('crypto.py').read(), \
    "This crypto.py is the OLD version (uses 'cryptography'). Upload the new pycryptodome-only crypto.py."
print("All files present:", os.listdir('.'))

## 3. Write buildozer.spec

In [ ]:
%%writefile /content/app/buildozer.spec
[app]
title = E2EE Chat
package.name = e2eechat
package.domain = org.e2ee
source.dir = .
source.include_exts = py,kv,json
source.exclude_dirs = venv,device_cache,cache_alice,cache_bob,__pycache__,.git,received_files,bin
source.exclude_patterns = server.py,database.py,test_*.py,*.db
version = 1.0
requirements = python3,kivy==2.3.0,pycryptodome
orientation = portrait
fullscreen = 0
android.permissions = INTERNET,ACCESS_NETWORK_STATE,READ_EXTERNAL_STORAGE,WRITE_EXTERNAL_STORAGE
android.api = 33
android.minapi = 24
android.archs = arm64-v8a
android.accept_sdk_license = True

[buildozer]
log_level = 2
warn_on_root = 0


## 4. Build the APK (20-40 min, keep this tab open)
Output is filtered to key lines; the full log is saved to `build.log`.

In [ ]:
%%bash
cd /content/app
yes | buildozer android debug 2>&1 | tee build.log | grep --line-buffered -E "^# |Downloading|Compiling|Packaging|Installing|BUILD|ERROR|Error|APK|Traceback"
echo "---- build finished ----"
ls -lh bin/ 2>/dev/null || { echo "NO APK PRODUCED - last 40 log lines:"; tail -n 40 build.log; }

## 5. Download the APK
Then copy it to your phone and install it (allow 'install unknown apps' when asked).

In [ ]:
import glob
from google.colab import files
apks = glob.glob('/content/app/bin/*.apk')
assert apks, "No APK found. Check the log above (tail -n 80 /content/app/build.log)."
files.download(apks[0])

## If the build fails
Run this cell and send the output to Claude:
```
!tail -n 80 /content/app/build.log
```
## After installing
Open the app, tap the three dots (top right) > **Config**, and enter your server PC's Wi-Fi IP and port 65432.